<a href="https://colab.research.google.com/github/vikkikumar1/ResumeIQ_AI/blob/main/jupyter_notebook/3.BERT_Finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, classification_report
from sklearn.metrics.pairwise import cosine_similarity
import json, warnings
warnings.filterwarnings('ignore')

print('Libraries imported!')

Libraries imported!


/tmp/ipykernel_8881/1846793969.py:5: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses
/tmp/ipykernel_8881/1846793969.py:6: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Load the cleaned ATS pairs dataset
df = pd.read_csv('/content/drive/MyDrive/resumeJD_Pairs_Version2.csv')
print(f'Loaded: {len(df)} pairs')
print(f'\nLabel distribution:')
print(df['match_label'].value_counts())
print(f'\nScore range: {df["match_score"].min():.2f} – {df["match_score"].max():.2f}')
df.head(3)


Loaded: 320 pairs

Label distribution:
match_label
high      118
low       115
medium     87
Name: count, dtype: int64

Score range: 0.00 – 0.95


,resume_text,job_description,match_score,match_label
0,Experienced Cloud Engineer with 3 years of exp...,We are hiring a Cloud Engineer. Required skill...,0.80,high
1,Experienced Full Stack Developer with 5 years ...,We are hiring a Full Stack Developer. Required...,0.83,high
2,Experienced Cloud Engineer with 3 years of exp...,We are hiring a Cloud Engineer. Required skill...,0.90,high


In [5]:
#Train / Val / Test Split
#Stratify by match_label so each split has balanced high/medium/low

train_df, temp_df = train_test_split(
    df, test_size=0.3, random_state=42, stratify=df['match_label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['match_label']
)

print(f'Train:      {len(train_df)} pairs')
print(f'Validation: {len(val_df)} pairs')
print(f'Test:       {len(test_df)} pairs')
print()

# Verify each split has all 3 labels
for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f'{name} label dist: {split["match_label"].value_counts().to_dict()}')

Train:      224 pairs
Validation: 48 pairs
Test:       48 pairs

Train label dist: {'high': 83, 'low': 80, 'medium': 61}
Val label dist: {'low': 18, 'high': 17, 'medium': 13}
Test label dist: {'high': 18, 'low': 17, 'medium': 13}


In [6]:
#Convert to InputExample format — sentence-transformers expects this
#text1 = resume, text2 = job description, label = match_score (float 0-1)

train_examples = [
    InputExample(texts=[row['resume_text'], row['job_description']], label=float(row['match_score']))
    for _, row in train_df.iterrows()
]

val_examples = [
    InputExample(texts=[row['resume_text'], row['job_description']], label=float(row['match_score']))
    for _, row in val_df.iterrows()
]

print(f'Train examples: {len(train_examples)}')
print(f'Val examples:   {len(val_examples)}')
print(f'\nSample InputExample:')
print(f'  text1 (resume): {train_examples[0].texts[0][:80]}...')
print(f'  text2 (JD):     {train_examples[0].texts[1][:80]}...')
print(f'  label:          {train_examples[0].label}')

Train examples: 224
Val examples:   48

Sample InputExample:
  text1 (resume): Experienced Cybersecurity Analyst with 9 years of experience in SIEM, SOC, Netwo...
  text2 (JD):     We are hiring a Cybersecurity Analyst. Required skills: SIEM, SOC, Network Secur...
  label:          0.04


In [7]:
#Base Model Baseline
print('Loading base BERT model...')
base_model = SentenceTransformer('all-mpnet-base-v2')
print('Loaded.')


Loading base BERT model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded.


In [11]:
df['resume_text'] = df['resume_text'].fillna('').astype(str)
df['job_description'] = df['job_description'].fillna('').astype(str)

In [13]:
test_df = test_df.copy()

test_df['resume_text'] = (
    test_df['resume_text']
    .fillna('')
    .astype(str)
)

test_df['job_description'] = (
    test_df['job_description']
    .fillna('')
    .astype(str)
)

In [14]:
# Evaluate base model on test set BEFORE fine-tuning
# This is our baseline, fine-tuning should beat this
print('Evaluating base model on test set...')

base_preds = []
for _, row in test_df.iterrows():
    emb1 = base_model.encode(row['resume_text'])
    emb2 = base_model.encode(row['job_description'])
    sim  = cosine_similarity([emb1], [emb2])[0][0]
    base_preds.append(float(sim))

base_mae  = mean_absolute_error(test_df['match_score'], base_preds)
base_rmse = np.sqrt(mean_squared_error(test_df['match_score'], base_preds))

print(f'\nBase Model — Test Set Performance')
print(f'  MAE:  {base_mae:.4f}')
print(f'  RMSE: {base_rmse:.4f}')
print()
print('Goal: fine-tuning should reduce MAE below this number.')

Evaluating base model on test set...

Base Model — Test Set Performance
  MAE:  0.3396
  RMSE: 0.4725

Goal: fine-tuning should reduce MAE below this number.


In [16]:
#Fine-Tune BERT

# Fresh model instance for fine-tuning (don't reuse base_model)
model = SentenceTransformer('all-mpnet-base-v2')

# DataLoader, shuffles training data each epoch
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

# CosineSimilarityLoss: trains the model so that
# cosine_similarity(embed(resume), embed(jd)) ≈ match_score
train_loss = losses.CosineSimilarityLoss(model)

# Evaluator runs on val set after each evaluation_steps
evaluator = EmbeddingSimilarityEvaluator.from_input_examples(
    val_examples, name='ats-val'
)

print('Training setup ready.')
print(f'  Batch size:   16')
print(f'  Train pairs:  {len(train_examples)}')
print(f'  Steps/epoch:  {len(train_dataloader)}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Training setup ready.
  Batch size:   16
  Train pairs:  224
  Steps/epoch:  14


In [22]:
import pandas as pd

# Clean train data
train_df['resume_text'] = train_df['resume_text'].fillna('').astype(str)
train_df['job_description'] = train_df['job_description'].fillna('').astype(str)
train_df['match_score'] = pd.to_numeric(train_df['match_score'], errors='coerce')

train_df = train_df.dropna(subset=['match_score'])

train_df = train_df[
    (train_df['resume_text'].str.strip() != '') &
    (train_df['job_description'].str.strip() != '')
]

print("Training rows:", len(train_df))

Training rows: 184


In [23]:
from sentence_transformers import InputExample

train_examples = []

for _, row in train_df.iterrows():

    train_examples.append(
        InputExample(
            texts=[
                row['resume_text'],
                row['job_description']
            ],
            label=float(row['match_score'])
        )
    )

print("Examples created:", len(train_examples))

Examples created: 184


In [25]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=16
)

print("Batches:", len(train_dataloader))

Batches: 12


In [26]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_df['resume_text'].astype(str).tolist(),
    sentences2=val_df['job_description'].astype(str).tolist(),
    scores=val_df['match_score'].astype(float).tolist()
)

print("Evaluator Ready")

Evaluator Ready


In [27]:
from sentence_transformers import losses

train_loss = losses.CosineSimilarityLoss(model)

print("Loss Function Ready")

Loss Function Ready


In [28]:
epochs = 10

total_steps = len(train_dataloader) * epochs
warmup_steps = int(total_steps * 0.1)

print("Total Steps:", total_steps)
print("Warmup Steps:", warmup_steps)

Total Steps: 120
Warmup Steps: 12


In [29]:
model.fit(
    train_objectives=[
        (train_dataloader, train_loss)
    ],
    evaluator=evaluator,
    epochs=epochs,
    evaluation_steps=len(train_dataloader),
    warmup_steps=warmup_steps,
    output_path="models/finetuned_resume_matcher",
    save_best_model=True,
    show_progress_bar=True
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Pearson Cosine,Spearman Cosine
12,No log,No log,-0.499183,-0.292654
24,No log,No log,-0.482992,-0.332690
36,No log,No log,-0.525705,-0.344385
48,No log,No log,-0.501211,-0.344820
60,No log,No log,-0.498432,-0.373052
72,No log,No log,-0.497433,-0.365001
84,No log,No log,-0.481629,-0.344602
96,No log,No log,-0.492140,-0.330895
108,No log,No log,-0.494819,-0.340305


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss,Pearson Cosine,Spearman Cosine
12,No log,No log,-0.499183,-0.292654
24,No log,No log,-0.482992,-0.332690
36,No log,No log,-0.525705,-0.344385
48,No log,No log,-0.501211,-0.344820
60,No log,No log,-0.498432,-0.373052
72,No log,No log,-0.497433,-0.365001
84,No log,No log,-0.481629,-0.344602
96,No log,No log,-0.492140,-0.330895
108,No log,No log,-0.494819,-0.340305
120,No log,No log,-0.494893,-0.341447


In [30]:
from sentence_transformers import SentenceTransformer

finetuned_model = SentenceTransformer(
    "models/finetuned_resume_matcher"
)

print("Fine-tuned model loaded")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fine-tuned model loaded


In [31]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

predictions = []

for _, row in test_df.iterrows():

    emb1 = finetuned_model.encode(
        str(row['resume_text'])
    )

    emb2 = finetuned_model.encode(
        str(row['job_description'])
    )

    sim = cosine_similarity(
        [emb1],
        [emb2]
    )[0][0]

    predictions.append(float(sim))

mae = mean_absolute_error(
    test_df['match_score'],
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        test_df['match_score'],
        predictions
    )
)

print("Fine-Tuned Results")
print("MAE :", round(mae, 4))
print("RMSE:", round(rmse, 4))

Fine-Tuned Results
MAE : 0.3311
RMSE: 0.4764


In [36]:
print("Base MAE :", base_mae)
print("Base RMSE:", base_rmse)

print("Fine-Tuned MAE :", mae)
print("Fine-Tuned RMSE:", rmse)

Base MAE : 0.33956219310561825
Base RMSE: 0.47247184904585215
Fine-Tuned MAE : 0.3311260775725047
Fine-Tuned RMSE: 0.47637071588080493


In [33]:
import os

print(os.path.exists("models/finetuned-bert"))

True


In [34]:
os.listdir("models/finetuned-bert")

['tokenizer.json',
 'sentence_bert_config.json',
 'config_sentence_transformers.json',
 'tokenizer_config.json',
 'config.json',
 'modules.json',
 '1_Pooling',
 '2_Normalize',
 'eval',
 'README.md',
 'model.safetensors']

In [41]:
import json
import os

os.makedirs('models/finetuned-bert', exist_ok=True)

metadata = {
    'base_model': 'all-mpnet-base-v2',
    'dataset': 'merged_dataset_clean.csv',
    'total_pairs': len(df),
    'train_pairs': len(train_df),
    'val_pairs': len(val_df),
    'test_pairs': len(test_df),
    'epochs': 10,
    'batch_size': 16,
    'base_mae': round(float(base_mae), 4),
    'finetuned_mae': round(float(mae), 4),
    'improvement_pct': round(
        ((base_mae - mae) / base_mae) * 100,
        2
    )
}

with open(
    'models/finetuned-bert/metadata.json',
    'w'
) as f:
    json.dump(metadata, f, indent=2)

print("Saved: models/finetuned-bert/")
print(json.dumps(metadata, indent=2))

Saved: models/finetuned-bert/
{
  "base_model": "all-mpnet-base-v2",
  "dataset": "merged_dataset_clean.csv",
  "total_pairs": 320,
  "train_pairs": 184,
  "val_pairs": 48,
  "test_pairs": 48,
  "epochs": 10,
  "batch_size": 16,
  "base_mae": 0.3396,
  "finetuned_mae": 0.3311,
  "improvement_pct": 2.48
}


In [42]:
#Production Pipeline Test

def score_resume_against_jd(resume_text, jd_text, model):

    emb_resume = model.encode(resume_text, convert_to_numpy=True)
    emb_jd     = model.encode(jd_text,     convert_to_numpy=True)
    score      = cosine_similarity([emb_resume], [emb_jd])[0][0]
    return float(score)


# Test with realistic cases
test_cases = [
    {
        'label':  'HIGH match expected',
        'resume': 'Senior Python developer with 6 years experience. Built REST APIs using FastAPI and Django. Proficient in PostgreSQL, Docker, and AWS deployment.',
        'jd':     'We need a Python backend engineer with FastAPI or Django experience. Must know SQL databases and cloud deployment.',
    },
    {
        'label':  'MEDIUM match expected',
        'resume': 'Frontend developer with 3 years React and TypeScript experience. Some Node.js and Express for small APIs.',
        'jd':     'Full stack engineer needed. React frontend required, strong Python backend preferred. 4+ years experience.',
    },
    {
        'label':  'LOW match expected',
        'resume': 'Executive Chef with 12 years experience in fine dining. Expert in French cuisine, menu planning, and kitchen management.',
        'jd':     'Data Scientist with Python, machine learning, and SQL experience required.',
    },
]

print('Production Pipeline Test')
print('=' * 60)
for case in test_cases:
    score = score_resume_against_jd(case['resume'], case['jd'], finetuned_model)
    label = 'HIGH' if score > 0.70 else 'MEDIUM' if score > 0.45 else 'LOW'
    print(f"\n{case['label']}")
    print(f"  Predicted score: {score:.4f}  →  {label}")


Production Pipeline Test

HIGH match expected
  Predicted score: 0.6824  →  MEDIUM

MEDIUM match expected
  Predicted score: 0.7645  →  HIGH

LOW match expected
  Predicted score: 0.2846  →  LOW


In [43]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [45]:
!cp -r models/finetuned-bert /content/drive/MyDrive/

In [46]:
!ls /content/drive/MyDrive/finetuned-bert

1_Pooling			   eval		      README.md
2_Normalize			   metadata.json      sentence_bert_config.json
config.json			   model.safetensors  tokenizer_config.json
config_sentence_transformers.json  modules.json       tokenizer.json


In [47]:
!zip -r finetuned-bert.zip models/finetuned-bert

  adding: models/finetuned-bert/ (stored 0%)
  adding: models/finetuned-bert/tokenizer.json (deflated 71%)
  adding: models/finetuned-bert/sentence_bert_config.json (deflated 43%)
  adding: models/finetuned-bert/config_sentence_transformers.json (deflated 40%)
  adding: models/finetuned-bert/tokenizer_config.json (deflated 48%)
  adding: models/finetuned-bert/metadata.json (deflated 41%)
  adding: models/finetuned-bert/config.json (deflated 48%)
  adding: models/finetuned-bert/modules.json (deflated 64%)
  adding: models/finetuned-bert/1_Pooling/ (stored 0%)
  adding: models/finetuned-bert/1_Pooling/config.json (deflated 17%)
  adding: models/finetuned-bert/2_Normalize/ (stored 0%)
  adding: models/finetuned-bert/eval/ (stored 0%)
  adding: models/finetuned-bert/eval/similarity_evaluation_results.csv (deflated 15%)
  adding: models/finetuned-bert/README.md (deflated 72%)
  adding: models/finetuned-bert/model.safetensors (deflated 8%)


In [48]:
!cp finetuned-bert.zip /content/drive/MyDrive/